# Lab 01-06 — Chunk-size sweep: retrieval quality vs chunk size

**Track 01 · Chunking** — the final chunking lab sweeps `chunk_size` and measures the RETRIEVAL impact — the classic chunk-size tradeoff. Small chunks are precise but fragment a passage across many vectors; large chunks are coherent but coarse (a whole passage may collapse into one chunk, dragging neighbouring topics along).

This notebook is **self-contained**: it imports LangChain splitters, `langchain_community.FAISS`, `HuggingFaceEmbeddings`, pandas, and tiktoken directly — no repo component library. Every block of the pipeline is built right here: the passage load, the recursive splitter, the BGE embedder, the FAISS index, and the self-recall probe all appear as plain code in the cells below, which is exactly how the shared components in `src/` work underneath.

```
passages.parquet -> RecursiveCharacterTextSplitter(size, overlap=50) -> BGE embed (local)
                 -> FAISS index (fresh per size) -> top-5 probe per derived query -> self-recall@5
Data   : Data/corpus/rag-mini-wikipedia/passages.parquet (subset: first 1000 passages)
Sweep  : chunk_size in [100, 200, 400, 800], overlap=50
Probe  : 50 passage-derived queries, top-5, self-recall@5, seed=42
```

Protocol: rag-mini-wikipedia has no qrels and its `test.parquet` answers are yes/no, so we use a passage-sourced **self-recall** probe instead. We take ~50 random passages, derive a query from each (its first sentence), retrieve top-5 chunks, and mark "source passage found" when any retrieved chunk belongs to that passage. Each chunk size gets a FRESH split + embed + FAISS index so the comparison is clean.

Embeddings are local BGE (`BAAI/bge-base-en-v1.5` via `HuggingFaceEmbeddings`, `normalize_embeddings=True` — which BGE requires for cosine).


## Setup

One prerequisite must hold before this notebook will run:

- **rag-mini-wikipedia on disk** — `Data/corpus/rag-mini-wikipedia/passages.parquet` (already fetched by the repo's manifest-verified fetchers).

No repo imports are needed: everything this notebook uses comes from `langchain-core`, `langchain-text-splitters`, `langchain-community`, `langchain-huggingface`, `pandas`, `tiktoken`, and `faiss-cpu`. The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. Unlike the Curriculum notebook, there is no `sys.path` trick: nothing is imported from `src/`.

The next cell installs the notebook-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt — the install
# below is a no-op if you have run `pip install -r requirements.txt`).
%pip install -q sentence-transformers langchain-huggingface langchain-community faiss-cpu langchain-text-splitters pandas tiktoken


In [ ]:
# Bootstrap: stdlib imports + repo-root walk (no sys.path tricks).
# Keep the sweep table the star of the output: silence the "Loading weights"
# progress bar (transformers honors HF_HUB_DISABLE_PROGRESS_BARS) and the
# deprecation notice langchain_community.FAISS logs for precomputed
# embeddings (a logging call, so set the module logger to ERROR). The env var
# must be set before any third-party import — langchain_text_splitters already
# pulls in huggingface_hub, which reads the flag at import time.
import os  # noqa: E402

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import logging  # noqa: E402
import random  # noqa: E402
import time  # noqa: E402
from pathlib import Path  # noqa: E402

logging.getLogger("langchain_community.vectorstores.faiss").setLevel(logging.ERROR)

# LangChain + pandas + tiktoken — the only libraries this notebook needs.
# Nothing is imported from the repo's src/ component library.
import pandas as pd  # noqa: E402
import tiktoken  # noqa: E402
from langchain_community.vectorstores import FAISS  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from langchain_core.embeddings import Embeddings  # noqa: E402
from langchain_huggingface import HuggingFaceEmbeddings  # noqa: E402
from langchain_text_splitters import RecursiveCharacterTextSplitter  # noqa: E402

# A notebook has no __file__, so walk up from the cwd to the repo root and
# cd into it — Data/... paths then resolve exactly like the lab script.
REPO_ROOT = Path.cwd()
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "curriculum").is_dir() and (_candidate / "NoteBooks").is_dir():
        REPO_ROOT = _candidate
        break
os.chdir(REPO_ROOT)


## 1. Configuration

`CORPUS_PATH` points at the rag-mini-wikipedia passages; `CORPUS_SUBSET = 1000` takes the first 1000 passages (keeps CPU embedding time reasonable); `CHUNK_SIZES` is the sweep grid; `OVERLAP` is shared across sizes; `PROBE_QUERIES = 50` bounds the probe set; `TOP_K = 5` is the retrieval depth for the self-recall check; `RANDOM_SEED = 42` fixes the probe set so runs are reproducible; `MODEL_NAME` is the local BGE embedder.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration — tweak these to re-run the experiment
# --------------------------------------------------------------------------
CORPUS_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
CORPUS_SUBSET = 1000  # first N passages — keeps CPU embedding time reasonable
CHUNK_SIZES = [100, 200, 400, 800]
OVERLAP = 50
PROBE_QUERIES = 50
TOP_K = 5
RANDOM_SEED = 42  # fixed seed so the probe set is reproducible
MODEL_NAME = "BAAI/bge-base-en-v1.5"


## 2. Load — passages + probe derivation

`load_passages` reads the first `limit` passages from the parquet as `Document`s carrying a `passage_id` metadata (the self-recall key). `derive_query` turns a passage into a probe query: the first ~40 words, trimmed at the last sentence-ending punctuation so the query is a complete, self-contained sentence. `preview` truncates text for printing.


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — passages + probe derivation
# --------------------------------------------------------------------------
def load_passages(path: Path, limit: int) -> list[Document]:
    """Load the first ``limit`` passages as Documents carrying a ``passage_id``."""
    df = pd.read_parquet(path)
    return [
        Document(page_content=text, metadata={"passage_id": i})
        for i, text in enumerate(df["passage"].head(limit).tolist())
    ]


def derive_query(text: str, max_words: int = 40) -> str:
    """Derive a probe query from a passage: first ~40 words, trimmed to a sentence.

    Cuts at the last sentence-ending punctuation inside the first ``max_words``
    so the query is a complete, self-contained sentence.
    """
    first = " ".join(text.split()[:max_words])
    cut = max((first.rfind(p) for p in ".!?"), default=-1)
    if cut > 0:
        first = first[: cut + 1]
    return first.strip() or text[:200].strip()


def preview(text: str, limit: int = 100) -> str:
    """Truncate text for printing (never dump full chunk contents)."""
    return text[:limit] + ("..." if len(text) > limit else "")


## 3. Measure one chunk size — split, embed, index, probe

The inline pipeline for a single chunk size, mirroring the repo's `FAISSVectorStore.add(chunks, embeddings=...)`: `RecursiveCharacterTextSplitter(size, OVERLAP)` splits the passages; the local BGE embedder embeds every chunk; the vectors are handed to `FAISS.from_documents` through a tiny precomputed passthrough (looked up BY TEXT, not by order — exactly what the repo's `_PrecomputedEmbeddings` does), so the embed step and the index step stay separately usable; and each probe query is embedded and searched with `similarity_search_by_vector` at `TOP_K`. `measure_chunk_size` returns one table row: chunk count, average token count (tiktoken), self-recall@5, and mean per-query latency in ms.


In [ ]:
# --------------------------------------------------------------------------
# 3. Measure one chunk size — split, embed, index, probe
# --------------------------------------------------------------------------
class _PrecomputedEmbeddings(Embeddings):
    """Hand the store precomputed vectors (looked up BY TEXT, not by order).

    FAISS calls embed_documents once with the full list; the lookup keeps the
    embed step and the index step separately timed, and would stay correct if
    a store batched the call.
    """

    def __init__(self, texts: list[str], embeddings: list[list[float]]):
        if len(texts) != len(embeddings):
            raise ValueError("texts and embeddings must be parallel lists")
        self._table: dict[str, list[float]] = dict(zip(texts, embeddings))

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        missing = [t for t in texts if t not in self._table]
        if missing:
            raise ValueError(f"{len(missing)} text(s) have no precomputed vector")
        return [self._table[t] for t in texts]

    def embed_query(self, text: str) -> list[float]:
        raise NotImplementedError("precomputed embeddings cannot embed queries")


def measure_chunk_size(
    size: int,
    passages: list[Document],
    probes: list[tuple[int, str]],
    embedder,
    enc,
) -> dict:
    """Split, embed, index and probe one chunk size; return one table row."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=OVERLAP)
    chunks = splitter.split_documents(passages)
    vectors = [list(v) for v in embedder.embed_documents(
        [c.page_content for c in chunks]
    )]

    store = FAISS.from_documents(
        chunks, embedding=_PrecomputedEmbeddings([c.page_content for c in chunks], vectors)
    )  # fresh index per chunk size

    avg_tokens = (
        sum(len(enc.encode(c.page_content)) for c in chunks) / len(chunks)
        if chunks
        else 0.0
    )

    found = 0
    latencies_ms: list[float] = []
    for passage_id, query in probes:
        query_vec = embedder.embed_query(query)
        t0 = time.perf_counter()
        hits = store.similarity_search_by_vector(query_vec, k=TOP_K)
        latencies_ms.append((time.perf_counter() - t0) * 1000)
        if any(h.metadata.get("passage_id") == passage_id for h in hits):
            found += 1

    return {
        "chunk_size": size,
        "n_chunks": len(chunks),
        "avg_tokens": avg_tokens,
        "recall": found / len(probes),
        "latency_ms": sum(latencies_ms) / len(latencies_ms),
    }


## 4. Experiment — the sweep

`run_experiment` runs the protocol: load the passage subset; derive the seed-fixed probe queries; then, for each chunk size, a FRESH split + embed + FAISS index and the 50-probe self-recall pass — four independent mini-pipelines, so the comparison is clean. The cell caps torch threads so the embedding sweep stays light on shared machines (probe latency is reported, not gated).


In [ ]:
# --------------------------------------------------------------------------
# 4. Experiment — the sweep
# --------------------------------------------------------------------------
def run_experiment() -> dict:
    # Keep the embedding sweep light on shared machines.
    try:
        import torch

        torch.set_num_threads(2)
    except ImportError:
        pass

    # --- 1. setup: what we are sweeping and how we measure it ---------------
    print("Chunk-size sweep — retrieval quality vs chunk size")
    print(f"  corpus      : {CORPUS_PATH} (subset: first {CORPUS_SUBSET} passages)")
    print(f"  embedder    : {MODEL_NAME} (local BGE, CPU)")
    print(f"  chunk sizes : {CHUNK_SIZES}  overlap={OVERLAP}")
    print(f"  probe       : {PROBE_QUERIES} passage-derived queries, "
          f"top-{TOP_K}, self-recall@{TOP_K}, seed={RANDOM_SEED}")
    print("  (first run downloads the BGE model, ~440MB)\n")

    # --- 2. load: a subset of passages as documents -------------------------
    passages = load_passages(CORPUS_PATH, CORPUS_SUBSET)
    avg_chars = sum(len(p.page_content) for p in passages) / len(passages)
    print(f"Loaded {len(passages)} passages "
          f"(avg {avg_chars:.0f} chars/passage)")

    # --- 3. probes: passage-derived queries (self-recall protocol) ----------
    rng = random.Random(RANDOM_SEED)
    probe_ids = rng.sample(range(len(passages)), PROBE_QUERIES)
    probes = [(pid, derive_query(passages[pid].page_content)) for pid in probe_ids]
    print(f"Derived {len(probes)} probe queries from random passages, e.g.:")
    for pid, query in probes[:2]:
        print(f"  [{pid}] {preview(query)!r}")

    # --- 4. sweep: fresh split + embed + index per chunk size ---------------
    embedder = HuggingFaceEmbeddings(
        model_name=MODEL_NAME,
        encode_kwargs={"normalize_embeddings": True},  # BGE needs cosine-normalized vectors
    )
    enc = tiktoken.encoding_for_model("gpt-4")
    print("\nSweeping chunk sizes (fresh index per size)...\n")
    rows = []
    for size in CHUNK_SIZES:
        rows.append(measure_chunk_size(size, passages, probes, embedder, enc))

    return {
        "passages": passages,
        "probes": probes,
        "rows": rows,
        "avg_chars": avg_chars,
    }


## 5. Demo — the artifact

`print_demo` renders the sweep table and the takeaway. The rows come straight from the experiment; the takeaway contrasts the extremes of the grid — `chunk_size=100` (many small chunks, fine granularity, highest per-query cost) vs `chunk_size=800` (few large chunks, coarse, cheapest to probe).


In [ ]:
# --------------------------------------------------------------------------
# 5. Demo — the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    rows: list[dict] = exp["rows"]

    print("--- chunk-size sweep: retrieval quality vs chunk size ---")
    header = (
        f"{'chunk_size':>10} | {'n_chunks':>9} | {'avg_tokens':>10} | "
        f"{'self_recall@5':>12} | {'mean_latency_ms':>15}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        print(f"{row['chunk_size']:>10} | {row['n_chunks']:>9} | "
              f"{row['avg_tokens']:>10.1f} | {row['recall']:>12.3f} | "
              f"{row['latency_ms']:>15.2f}")

    print("\n--- takeaway: chunk size trades granularity against coherence ---")
    small, large = rows[0], rows[-1]
    print(
        f"chunk_size {small['chunk_size']} -> {large['chunk_size']}: "
        f"{small['n_chunks']} -> {large['n_chunks']} chunks "
        f"({small['n_chunks'] / large['n_chunks']:.1f}x fewer), "
        f"avg {small['avg_tokens']:.0f} -> {large['avg_tokens']:.0f} "
        "tokens/chunk"
    )
    print(
        "self-recall@5 stays high at every size because the probes are cut "
        "verbatim from passage openings — near-duplicate queries that any "
        "index finds. The real costs show in the other columns:"
    )
    print(
        "  * small chunks -> many vectors: more index memory, more candidates "
        "per query (latency), and a mid-passage fact gets torn across chunks, "
        "so retrieval returns pieces instead of the answer."
    )
    print(
        "  * large chunks -> few, fat vectors: fast and coherent, but coarse — "
        "a whole passage may be one chunk, so a query pulls in neighbouring "
        "topics and the context window fills with irrelevant text."
    )
    print(
        "The sweet spot is the smallest chunk size that still keeps each "
        "passage's facts inside one retrievable unit."
    )


## 6. Verification gate

`verify_gate` checks the concrete, measurable claims of this lab:

1. **the sweep ran at every size** — one row per `CHUNK_SIZES` entry;
2. **smaller chunk size -> more chunks** — `n_chunks` decreases monotonically as size grows (splitting a corpus at 100 chars cannot produce fewer pieces than at 800);
3. **larger chunk size -> longer chunks** — average token count per chunk increases monotonically;
4. **retrieval worked everywhere** — self-recall is strictly inside (0, 1] at every size, and probe latency is positive;
5. **the probe set is the declared size** — exactly `PROBE_QUERIES` queries.

Everything is measured from the experiment output, never asserted.


In [ ]:
# --------------------------------------------------------------------------
# 6. Verification gate
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    rows: list[dict] = exp["rows"]
    failures = 0

    def check(ok: bool, msg: str) -> None:
        nonlocal failures
        if ok:
            print(f"  [PASS] {msg}")
        else:
            print(f"  [FAIL] {msg}")
            failures += 1

    # 1. the sweep ran at every size
    check(
        len(rows) == len(CHUNK_SIZES),
        f"sweep produced one row per chunk size: {len(rows)} rows "
        f"(expected {len(CHUNK_SIZES)})",
    )

    # 2. smaller chunk size -> more chunks (monotonic)
    sizes = [r["chunk_size"] for r in rows]
    n_chunks = [r["n_chunks"] for r in rows]
    check(
        n_chunks == sorted(n_chunks, reverse=True),
        f"n_chunks decreases with chunk size: {dict(zip(sizes, n_chunks))}",
    )

    # 3. larger chunk size -> longer chunks (monotonic)
    avg_tokens = [r["avg_tokens"] for r in rows]
    check(
        avg_tokens == sorted(avg_tokens),
        f"avg_tokens increases with chunk size: "
        f"{[round(t, 1) for t in avg_tokens]}",
    )

    # 4. retrieval worked everywhere — recall in (0, 1] and latency > 0
    recalls = [r["recall"] for r in rows]
    latencies = [r["latency_ms"] for r in rows]
    check(
        all(0 < r <= 1 for r in recalls),
        f"self-recall in (0, 1] at every size: {[round(r, 3) for r in recalls]}",
    )
    check(
        all(l > 0 for l in latencies),
        f"probe latency positive at every size: "
        f"{[round(l, 1) for l in latencies]} ms",
    )

    # 5. the probe set is the declared size
    check(
        len(exp["probes"]) == PROBE_QUERIES,
        f"probe set has {len(exp['probes'])} queries (expected {PROBE_QUERIES})",
    )

    if failures:
        raise AssertionError(f"verification gate failed: {failures} check(s)")
    return 0


## Run the experiment

Run the sweep: load the passages, derive the probes, and sweep `chunk_size` with a fresh index per size.


In [ ]:
exp = run_experiment()


### Demo — the artifact

Render the sweep table and the granularity-vs-coherence takeaway.


In [ ]:
print_demo(exp)


### Verification gate

Check the concrete claims of the experiment — sweep coverage, monotonicity in both directions, working retrieval everywhere, and the declared probe size.


In [ ]:
verify_gate(exp)
